# Inflation curves (CPI / breakeven and real rates)

Deep-dive reference: dedicated `InflationCurve` bindings for CPI paths, observation lags, and inflation-linked pricing intuition.

## Concept

**Inflation curves** describe expected **CPI** growth or **inflation swap** breakevens, used to price **TIPS**, **ILBs**, and **inflation swaps**. A common identity (Fisher-style, continuous-time shorthand) links **nominal** \(r_n\), **real** \(r_r\), and **inflation** \(\pi\): \(r_n \approx r_r + \pi\). When no dedicated curve type exists, you can still illustrate **real zero rates** from a **nominal** curve and an assumed **inflation** path.

## API walkthrough

`InflationCurve` is available directly from `finstack.core.market_data`. Build it from CPI knot points, query raw CPI levels with `cpi(t)`, apply the contractual observation lag with `cpi_with_lag(t)`, and compute annualized inflation between horizons with `inflation_rate(t1, t2)`.

In [1]:
from datetime import date

from finstack.core.market_data import InflationCurve

curve = InflationCurve(
    "USD-CPI",
    date(2025, 1, 15),
    315.5,
    [(0.0, 315.5), (1.0, 323.4), (2.0, 331.8), (5.0, 357.9), (10.0, 405.4)],
    indexation_lag_months=3,
    interp="log_linear",
)
print("curve:", curve)
print("base_cpi:", curve.base_cpi, "lag_months:", curve.indexation_lag_months)
for t in (1.0, 2.0, 5.0):
    print(f"t={t}y  cpi={curve.cpi(t):.3f}  cpi_with_lag={curve.cpi_with_lag(t):.3f}")
print("5Y annualized inflation:", round(curve.inflation_rate(0.0, 5.0), 6))

curve: InflationCurve(id="USD-CPI")
base_cpi: 315.5 lag_months: 3
t=1.0y  cpi=323.400  cpi_with_lag=321.407
t=2.0y  cpi=331.800  cpi_with_lag=329.680
t=5.0y  cpi=357.900  cpi_with_lag=355.649
5Y annualized inflation: 0.02554


## Practical example

**Nominal OIS** discount curve plus a **constant CPI breakeven** (illustrative) to derive **approximate real zeros** with \(r_r \approx r_n - \pi\).

**Caveats:** Real **CPI** is **lagged** (release delay and contract **observation lags**), **seasonal** (and often revised), and market breakevens embed an **inflation risk premium** and liquidity effects—so a flat \(\pi\) is only a teaching shortcut. In **continuous compounding**, if nominal and real zeros are defined consistently, \(DF(t) \approx e^{-r t}\) links discounting to zeros; a Fisher-style split \(r_n \approx r_r + \pi\) is still an approximation and must use compatible conventions in production.

In [2]:
base_cpi = curve.base_cpi
notional = 1_000_000.0
for t in (1.0, 3.0, 5.0, 10.0):
    ratio = curve.cpi_with_lag(t) / base_cpi
    adjusted_principal = notional * ratio
    print(
        f"t={t:>4.1f}y  lagged_index_ratio={ratio:.6f}  inflation_adjusted_notional={adjusted_principal:,.2f}"
    )

t= 1.0y  lagged_index_ratio=1.018722  inflation_adjusted_notional=1,018,721.54
t= 3.0y  lagged_index_ratio=1.071762  inflation_adjusted_notional=1,071,761.98
t= 5.0y  lagged_index_ratio=1.127254  inflation_adjusted_notional=1,127,254.28
t=10.0y  lagged_index_ratio=1.276963  inflation_adjusted_notional=1,276,962.90


## Takeaways

- `InflationCurve` gives you direct access to CPI levels, lag-adjusted CPI, and annualized inflation over arbitrary horizons.
- Observation lag matters: `cpi_with_lag(t)` is the right conceptual hook for linker and inflation-swap settlement mechanics.
- Calibrated inflation curves still depend on discounting, seasonality choices, and instrument conventions, so treat the simple examples here as structure-first rather than full production setup.

## Calibration from inflation swap quotes

With the calibration engine, we can bootstrap an **inflation curve** from zero-coupon inflation swap (ZCIS) rates.  This requires:
1. A pre-existing **discount curve** (for PV calculations)
2. ZCIS quotes at various maturities with a **base CPI** level and **observation lag**

The engine solves for the CPI projection path that reprices each ZCIS to zero NPV.

In [3]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.valuations import calibrate

def tenor(count, unit):
    return {"tenor": {"count": count, "unit": unit}}

# First build a discount curve as initial_market
dc = DiscountCurve(
    "USD-OIS", date(2025, 1, 15),
    [(0.0, 1.0), (1.0, 0.9478), (3.0, 0.8726), (5.0, 0.8099), (10.0, 0.6681)],
    day_count="act_365f",
)
initial_ctx = MarketContext()
initial_ctx.insert(dc)
initial_market_json = json.loads(initial_ctx.to_json())

plan = {
    "schema": "finstack.calibration/2",
    "plan": {
        "id": "us-cpi-curve",
        "description": "US CPI inflation curve from ZCIS quotes",
        "quote_sets": {
            "zcis": [
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2026-01-15", "rate": 0.0250, "index": "USA-CPI-U", "convention": "USD-CPI"}},
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2028-01-15", "rate": 0.0260, "index": "USA-CPI-U", "convention": "USD-CPI"}},
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2030-01-15", "rate": 0.0255, "index": "USA-CPI-U", "convention": "USD-CPI"}},
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2035-01-15", "rate": 0.0245, "index": "USA-CPI-U", "convention": "USD-CPI"}},
            ],
        },
        "steps": [{
            "id": "USD-CPI",
            "quote_set": "zcis",
            "kind": "inflation",
            "curve_id": "USD-CPI",
            "currency": "USD",
            "base_date": "2025-01-15",
            "discount_curve_id": "USD-OIS",
            "index": "USA-CPI-U",
            "observation_lag": "3M",
            "base_cpi": 315.5,
            "method": "Bootstrap",
            "interpolation": "linear",
        }],
        "settings": {"use_parallel": False},
    },
    "initial_market": initial_market_json,
}

result = calibrate(json.dumps(plan))
print("Success:", result.success)
print(result.report_to_dataframe().to_string(index=False))
print()

cal_curve = result.market.get_inflation_curve("USD-CPI")
print("Calibrated CPI path:")
for t in (1.0, 3.0, 5.0, 10.0):
    print(
        f"  t={t:>4.1f}y  cpi={cal_curve.cpi(t):.3f}  annualized_inflation={cal_curve.inflation_rate(0.0, t):.6f}"
    )

Success: True
step_id  success  iterations  max_residual         rmse                                                                          convergence_reason
USD-CPI     True         518  1.610542e-13 1.337833e-13 generic bootstrap calibration converged: max_residual (1.61e-13) within tolerance (1.00e-8)

Calibrated CPI path:
  t= 1.0y  cpi=325.719  annualized_inflation=0.032390
  t= 3.0y  cpi=343.304  annualized_inflation=0.028553
  t= 5.0y  cpi=360.716  annualized_inflation=0.027148
  t=10.0y  cpi=403.309  annualized_inflation=0.024858
